In [1]:
import os
import tensorflow as tf
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from tensorflow.keras import layers, models
from loss_functions.loss_nll import custom_loss

import OptimizedDataGenerator_v2 as DG
#from loss_functions.loss_hybrid3 import * # this one matters
#from loss_functions.loss_nll import *
from loss_functions.Slim_SSE import *
#from models.models import *
from models.Slim1D_model_Compression_IQ import *
from models.SoftQuantizeLayer import *
import utils
pi = 3.14159265359

maxval=1e9
minval=1e-9

2025-08-27 16:19:20.121839: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-27 16:19:20.121912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-27 16:19:20.122955: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-27 16:19:20.130266: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-27 16:19:23.381167: W tensorflow/compiler/tf2

In [2]:

fingerprint = 'd5cfc41b'
# c9ce48b7

input_shape = (16, 16, 2)
model = CreateSlim1DModel(
    input_shape=input_shape,
    
    # ========== SOFT INPUT QUANTIZATION PARAMETERS ==========
    use_soft_input_quantization=True,
    soft_quant_n_bits=2,
    soft_quant_initial_range=[-1.0, 1.0],
    soft_quant_trainable_levels=False,
    soft_quant_trainable_bins=True,
    soft_quant_initial_k=1.0,
    soft_quant_trainable_k=True,
    
    # ========== MODEL SIZE PARAMETERS ==========
    conv_filters=5,           # Conv1D filters (5 original)
    conv_kernel_size=3,       # Conv1D kernel size (3 original)
    dense1_units=12,          # First dense layer size (16 original)
    dense2_units=12,          # Second dense layer size (16 original)
    output_units=3,            # Output size (keep at 3 for x,y,β)
    
    # ========== QUANTIZATION PARAMETERS ==========
    use_quantization=False,   # True/False to enable QKeras
    conv_weight_bits=4,       # Conv1D weight precision (4 like baseline)
    conv_activation_bits=4,   # Conv1D activation precision (4 like baseline)
    dense_weight_bits=8,      # Dense weight precision (8 like baseline)
    dense_activation_bits=8,  # Dense activation precision (8 like baseline)
    
    # ========== PRUNING PARAMETERS ==========
    use_pruning=False,        # True/False to enable pruning
    pruning_fraction=0.1,     # 0.2=20%, 0.5=50%, 0.8=80% weights removed
    pruning_schedule='constant'  # 'constant' or 'polynomial_decay'

    

)
'''
model = CreateSlim1DModel(
    input_shape=(16, 16, 2))
'''
base_dir = f'./trained_models/model-{fingerprint}-checkpoints/'
checkpoint_files = [os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.hdf5')]
latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
training_cp_path = os.path.join(base_dir, 'training_log.csv')
training_history = pd.read_csv(training_cp_path)

print(f"Loading model from {latest_checkpoint}")
model.load_weights(latest_checkpoint)


Loading model from ./trained_models/model-d5cfc41b-checkpoints/weights.99-t102.76-v100.52.hdf5


ValueError: Layer count mismatch when loading weights from file. Model expected 6 layers, found 5 saved layers.

In [ ]:
plt.scatter(training_history['epoch'], training_history['loss'])
plt.scatter(training_history['epoch'], training_history['val_loss'])
plt.legend(['training', 'validation'])
plt.grid(True)
plt.xlabel('Epochs')
plt.ylabel('NLL loss')
plt.tight_layout() 

#plt.savefig(os.path.join(base_dir,'training_hist.png'))
plt.show()

In [ ]:
sq_layer = model.get_layer(name="soft_quantizer_output")
print("Current k:", sq_layer.k.numpy())
print("sq_layer:", sq_layer.levels)
print("sq_layer", sq_layer.bin_centers)

x_input = tf.constant(np.linspace(-1.5, 1.5, 500), dtype=tf.float32)
initial_first_level = sq_layer.first_level.numpy()[0]
initial_level_deltas = tf.exp(sq_layer.log_level_deltas).numpy()
initial_first_bin = sq_layer.first_bin_center.numpy()[0]
initial_bin_deltas = tf.exp(sq_layer.log_bin_deltas).numpy()

y_hard = sq_layer._hard_quantize(x_input, sq_layer.levels, sq_layer.bin_centers)
y_soft = sq_layer._soft_quantize(x_input, sq_layer.levels, sq_layer.bin_centers)

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(x_input, x_input, 'k--', alpha=0.4, label='Identity ($y=x$)')
ax.plot(x_input, y_hard, color='red', linewidth=2.5, label=f'Hard Quantize (Inference)')
ax.plot(x_input, y_soft, 'blue', linestyle='-', linewidth=2.5, label=f'Soft Quantize (Deriv. Apprx.)')

ax.set_title("SoftQuantizeLayer Behavior Verification", fontsize=16)
ax.set_xlabel("Input Value", fontsize=12)
ax.set_ylabel("Output Value", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)

plt.show()

In [ ]:
print(training_history.to_string())


In [ ]:
batch_size = 5000
test_batch_size = 5000
test_file_size = 20
data_directory_path = "/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/contained/test"
tfrecords_dir_validation = "/depot/cms/users/kuang14/Smart_Pixel/tf_records/16x16test_2"
test_generator = DG.OptimizedDataGenerator(
    # Base directory where your parquet files are located
    dataset_base_dir = data_directory_path,
    is_directory_recursive = False,
    batch_size = test_batch_size,
    file_count = test_file_size,
    to_standardize = True,
    include_y_local = False,
    # Labels you want to extract
    labels_list = ['x-midplane', 'y-midplane', 'cotBeta'], 
    # Input shape for the model (time slices, height, width)
    input_shape = (2, 16, 16),
    transpose = (0, 2, 3, 1),
    shuffle = False,
    files_from_end = False,
    
    # New parameters specific to this generator
    tfrecords_dir = tfrecords_dir_validation,
    use_time_stamps = [0, 19],  # Specific time stamps to use
    quantize = False,  # Whether to use quantization during loading
    max_workers = 2  # For parallel processing
)

In [ ]:
model.summary()

In [ ]:
# predicts test data
p_test = model.predict(test_generator)

In [ ]:
complete_truth = None
for _, y in tqdm(test_generator):
        if complete_truth is None:
            complete_truth = y
        else:
            complete_truth = np.concatenate((complete_truth, y), axis=0)

In [ ]:
df = pd.DataFrame(p_test, columns=['x', 'y', 'cotB'])

# Store true values (only 3 for slim model)
df['xtrue'] = complete_truth[:,0]
df['ytrue'] = complete_truth[:,1] 
df['cotBtrue'] = complete_truth[:,2]  # Note: cotB is now at index 2

# Calculate residuals
df['x_residual'] = df['xtrue'] - df['x']
df['y_residual'] = df['ytrue'] - df['y']
df['cotB_residual'] = df['cotBtrue'] - df['cotB']

# ADD THESE LINES for the metric calculation cells:
residuals = df['x_residual']    # For X metrics
residualsy = df['y_residual']   # For Y metrics  
residualsB = df['cotB_residual'] # For CotB metrics

# Save results
df.to_csv("test_slim_model.csv", header=True, index=False)

### Define Scaling factors first

In [ ]:
test_generator.labels_scale

In [ ]:
SCALE_FACTOR_X = 99.1550028      # Scaling factor for x position [μm]
SCALE_FACTOR_Y = 24.77847354  # Scaling factor for y position [μm]
#SCALE_FACTOR_COTA = 8.95462059   # Scaling factor for cotangent alpha
SCALE_FACTOR_COTB = 1.91059443 # Scaling factor for cotangent beta
#array([99.1550028 , 24.77847354,  8.95462059,  4.12538643])

# residuals (mean, stdv)

In [ ]:
#before sf

In [ ]:
(np.mean(residuals), np.std(residuals))

In [ ]:
(np.mean(residualsy), np.std(residualsy))

In [ ]:
(np.mean(residualsB), np.std(residualsB))

In [ ]:
(np.mean(residuals)*SCALE_FACTOR_X, np.std(residuals)*SCALE_FACTOR_X)

In [ ]:
(np.mean(residualsy)*SCALE_FACTOR_Y, np.std(residualsy)*SCALE_FACTOR_Y)

In [ ]:
(np.mean(residualsB)*SCALE_FACTOR_COTB, np.std(residualsB)*SCALE_FACTOR_COTB)

In [ ]:
# Convert cotBeta values to degrees
df['cotBtrue_deg'] = np.degrees(np.arctan(1.0 / df['cotBtrue']))
df['cotB_deg'] = np.degrees(np.arctan(1.0 / df['cotB']))
df['cotB_deg_residual'] = df['cotBtrue_deg'] - df['cotB_deg']

# Update the residuals variable for degree version
residualsB_deg = df['cotB_deg_residual']

# Now run the asymmetry test again with the new degree residuals
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

parameters = [('x_residual', residuals), ('y_residual', residualsy), ('cotB_deg_residual', residualsB_deg)]

for i, (name, data) in enumerate(parameters):
    # Plot histogram
    axes[i].hist(data, bins=50, alpha=0.7, density=True, edgecolor='black')
    
    # Calculate and show key statistics
    mean_val = np.mean(data)
    median_val = np.median(data)
    
    # Calculate percentiles for asymmetry check
    p16 = np.percentile(data, 16)
    p84 = np.percentile(data, 84)
    
    lower_width = median_val - p16
    upper_width = p84 - median_val
    asymmetry_ratio = upper_width / lower_width if lower_width != 0 else float('inf')
    
    # Add vertical lines
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.3f}')
    axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.3f}')
    axes[i].axvline(p16, color='orange', linestyle=':', alpha=0.7)
    axes[i].axvline(p84, color='orange', linestyle=':', alpha=0.7)
    
    axes[i].set_title(f'{name}\nAsymmetry ratio: {asymmetry_ratio:.2f}')
    axes[i].set_xlabel('Residual value')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    
    # Print numerical summary
    print(f"\n{name} asymmetry analysis:")
    print(f"  Mean: {mean_val:.4f}, Median: {median_val:.4f}")
    print(f"  Lower width (med-p16): {lower_width:.4f}")
    print(f"  Upper width (p84-med): {upper_width:.4f}")
    print(f"  Asymmetry ratio (upper/lower): {asymmetry_ratio:.3f}")
    if asymmetry_ratio > 1.2 or asymmetry_ratio < 0.8:
        print(f"  -> {name} shows significant asymmetry!")

plt.tight_layout()
plt.show()

In [ ]:
# Correct cotBeta to Beta degree conversion [0, 180] range
def cotbeta_to_beta_degrees_correct(cotbeta_values):
    """Convert cotBeta to Beta in degrees, ensuring [0, 180] range"""
    beta_rad = np.arctan(1.0 / cotbeta_values)
    beta_deg = np.degrees(beta_rad)
    
    # Ensure angles are in [0, 180] range as David specified
    beta_deg = np.where(beta_deg < 0, beta_deg + 180, beta_deg)
    
    return beta_deg

# Apply the corrected conversion
df['beta_true_deg'] = cotbeta_to_beta_degrees_correct(df['cotBtrue'])
df['beta_pred_deg'] = cotbeta_to_beta_degrees_correct(df['cotB'])
df['beta_deg_residual'] = df['beta_true_deg'] - df['beta_pred_deg']

# Check the results
residualsB_deg = df['beta_deg_residual']
print(f"Beta degree residuals (corrected to [0,180] range):")
print(f"  Mean: {np.mean(residualsB_deg):.4f}")
print(f"  Median: {np.median(residualsB_deg):.4f}")
print(f"  Std: {np.std(residualsB_deg):.4f}")
print(f"  Range: {np.min(residualsB_deg):.2f} to {np.max(residualsB_deg):.2f}")

print(f"\nBeta true range: {df['beta_true_deg'].min():.2f} to {df['beta_true_deg'].max():.2f}")
print(f"Beta pred range: {df['beta_pred_deg'].min():.2f} to {df['beta_pred_deg'].max():.2f}")

# Now run asymmetry test with corrected Beta degree residuals
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

parameters = [('x_residual', residuals), ('y_residual', residualsy), ('beta_deg_residual', residualsB_deg)]

for i, (name, data) in enumerate(parameters):
    # Plot histogram
    axes[i].hist(data, bins=50, alpha=0.7, density=True, edgecolor='black')
    
    # Calculate and show key statistics
    mean_val = np.mean(data)
    median_val = np.median(data)
    
    # Calculate percentiles for asymmetry check
    p16 = np.percentile(data, 16)
    p84 = np.percentile(data, 84)
    
    lower_width = median_val - p16
    upper_width = p84 - median_val
    asymmetry_ratio = upper_width / lower_width if lower_width != 0 else float('inf')
    
    # Add vertical lines
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.3f}')
    axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.3f}')
    axes[i].axvline(p16, color='orange', linestyle=':', alpha=0.7)
    axes[i].axvline(p84, color='orange', linestyle=':', alpha=0.7)
    
    axes[i].set_title(f'{name}\nAsymmetry ratio: {asymmetry_ratio:.2f}')
    axes[i].set_xlabel('Residual value')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    
    # Print numerical summary
    print(f"\n{name} asymmetry analysis:")
    print(f"  Mean: {mean_val:.4f}, Median: {median_val:.4f}")
    print(f"  Lower width (med-p16): {lower_width:.4f}")
    print(f"  Upper width (p84-med): {upper_width:.4f}")
    print(f"  Asymmetry ratio (upper/lower): {asymmetry_ratio:.3f}")
    if asymmetry_ratio > 1.2 or asymmetry_ratio < 0.8:
        print(f"  -> {name} shows significant asymmetry!")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def shortest_interval_68(data):
    """
    Compute the shortest interval that contains 68% of the data.
    """
    data = np.sort(data)
    n = len(data)
    ci_size = int(np.floor(0.68 * n))  # number of points in 68% interval
    min_width = float("inf")
    min_i = 0
    
    # Slide a window of size ci_size to find the narrowest range
    for i in range(n - ci_size):
        width = data[i + ci_size] - data[i]
        if width < min_width:
            min_width = width
            min_i = i
    
    # Extract interval bounds and median
    low = data[min_i]
    high = data[min_i + ci_size]
    median = np.median(data)
    
    return {
        "median": median,
        "error_low": median - low,    # asymmetric lower error
        "error_high": high - median   # asymmetric upper error
    }

# Test the shortest interval method on all three parameters using df columns
parameters = [
    ('x_residual', df['x_residual']), 
    ('y_residual', df['y_residual']), 
    ('beta_deg_residual', df['beta_deg_residual'])
]

print("Shortest 68% Interval Analysis:")
print("=" * 50)

for name, data in parameters:
    # Get shortest interval results
    result = shortest_interval_68(data)
    
    # Calculate asymmetry ratio
    asymmetry_ratio = result['error_high'] / result['error_low'] if result['error_low'] != 0 else float('inf')
    
    print(f"\n{name}:")
    print(f"  Median: {result['median']:.4f}")
    print(f"  Error low:  -{result['error_low']:.4f}")
    print(f"  Error high: +{result['error_high']:.4f}")
    print(f"  Asymmetry ratio (high/low): {asymmetry_ratio:.3f}")
    
    # Compare to standard percentile method
    p16 = np.percentile(data, 16)
    p84 = np.percentile(data, 84)
    median_std = np.median(data)
    std_low = median_std - p16
    std_high = p84 - median_std
    std_asymmetry = std_high / std_low if std_low != 0 else float('inf')
    
    print(f"  [Compare] Standard percentile asymmetry: {std_asymmetry:.3f}")
    
    if asymmetry_ratio > 1.2 or asymmetry_ratio < 0.8:
        print(f"  -> {name} shows significant asymmetry with shortest interval!")
    elif abs(asymmetry_ratio - std_asymmetry) > 0.05:
        print(f"  -> Shortest interval reveals different asymmetry than percentiles")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Focus on beta residuals
beta_data = df['beta_deg_residual']

# Get both methods' results
beta_shortest = shortest_interval_68(beta_data)
p16 = np.percentile(beta_data, 16)
p84 = np.percentile(beta_data, 84)
median_val = np.median(beta_data)

# Calculate bounds for both methods
shortest_low = beta_shortest['median'] - beta_shortest['error_low']
shortest_high = beta_shortest['median'] + beta_shortest['error_high']

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot histogram
counts, bins, patches = ax.hist(beta_data, bins=60, alpha=0.7, density=True, 
                               edgecolor='black', color='lightblue')

# Mark median
ax.axvline(beta_shortest['median'], color='green', linewidth=2, 
           linestyle='--', label=f'Median: {beta_shortest["median"]:.3f}°')

# Mark standard percentiles (16th, 84th)
ax.axvline(p16, color='orange', linewidth=2, linestyle=':', 
           label='Standard 68% (16th, 84th percentiles)')
ax.axvline(p84, color='orange', linewidth=2, linestyle=':')

# Mark shortest interval bounds
ax.axvline(shortest_low, color='red', linewidth=2, linestyle='-', 
           label='Shortest 68% interval')
ax.axvline(shortest_high, color='red', linewidth=2, linestyle='-')

# Shade the shortest interval
ax.axvspan(shortest_low, shortest_high, alpha=0.2, color='red')

# Add text annotations
ax.text(0.02, 0.95, f'Shortest Interval Method:', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', verticalalignment='top')
ax.text(0.02, 0.90, f'Lower error: {beta_shortest["error_low"]:.3f}°', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')
ax.text(0.02, 0.85, f'Upper error: {beta_shortest["error_high"]:.3f}°', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')
ax.text(0.02, 0.80, f'Asymmetry ratio: {beta_shortest["error_high"]/beta_shortest["error_low"]:.3f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.text(0.02, 0.70, f'Standard Percentile Method:', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', verticalalignment='top')
ax.text(0.02, 0.65, f'Lower error: {median_val - p16:.3f}°', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')
ax.text(0.02, 0.60, f'Upper error: {p84 - median_val:.3f}°', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')
ax.text(0.02, 0.55, f'Asymmetry ratio: {(p84 - median_val)/(median_val - p16):.3f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top')

ax.set_xlabel('Beta Residual (degrees)')
ax.set_ylabel('Density')
ax.set_title('Beta Degree Residuals: Asymmetric vs Symmetric Error Analysis')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print the practical difference
print("Practical Impact:")
print(f"Standard method would report: β ± {(p84-p16)/2:.2f}°")
print(f"Asymmetric method reports: β +{beta_shortest['error_high']:.2f}/-{beta_shortest['error_low']:.2f}°")
print(f"Difference in upper bound: {beta_shortest['error_high'] - (p84-p16)/2:.3f}°")

In [ ]:
def residual_plot_slim(ax, thisdf, var1, var2, name, scaling=1.0):
    """
    Residual plot for slim model (no uncertainty estimates)
    Creates binned residual plot with statistical uncertainty bands
    """
    
    nbins = 15
    
    var1_scaled = thisdf[var1] * scaling
    var2_scaled = thisdf[var2] * scaling
    residual_scaled = var1_scaled - var2_scaled
    
    xmin = np.min(var1_scaled)
    xmax = np.max(var1_scaled)
    
    step = 1.0*(xmax-xmin)/nbins
    
    # Create binned scatter plot
    x = sns.regplot(x=var1_scaled, y=residual_scaled, x_bins=np.linspace(xmin,xmax,nbins), 
                   fit_reg=None, marker='.', ax=ax, scatter_kws={'s': 20})
    ax.set_xlabel('True ' + name)
    ax.set_ylabel('True - predicted ' + name)
    
    # Calculate binned statistics for uncertainty bands
    bin_centers = []
    means = []
    stds = []
    
    for i in range(nbins):
        bin_mask = (var1_scaled >= xmin + i*step) & (var1_scaled < xmin + (i+1)*step)
        if np.sum(bin_mask) > 0:  # Only if there are points in this bin
            bin_centers.append(xmin + (i+0.5)*step)
            bin_residuals = residual_scaled[bin_mask]
            means.append(np.mean(bin_residuals))
            stds.append(np.std(bin_residuals))
    
    bin_centers = np.array(bin_centers)
    means = np.array(means)
    stds = np.array(stds)
    
    # Create uncertainty bands (±1 standard deviation)
    upbar = means + stds
    downbar = means - stds
    
    # Fill between for uncertainty band
    ax.fill_between(bin_centers, upbar, downbar, alpha=0.3, color='lightblue')
    
    return ax

def residual_plot_deg_slim(ax, thisdf, var1, var2, name, scaling=1.0):
    """
    Residual plot for cotangent values converted to degrees
    """
    # If not a cotangent variable, use regular residual plot
    if 'cot' not in var1:
        return residual_plot_slim(ax, thisdf, var1, var2, name, scaling=scaling)

    # Convert cotangent to degrees
    def inverse_cot(cota):
        a = np.arctan(1.0/cota)
        a = np.where(a < 0, a + np.pi, a)  # Using np.where for array operations
        return a    

    # Create temporary copies for degree conversion
    thisdf_copy = thisdf.copy()
    
    # Convert predictions to degrees
    thisdf_copy['angle'] = inverse_cot(thisdf_copy[var2].values * scaling) * 180 / np.pi
    
    # Convert true values to degrees  
    thisdf_copy['angletrue'] = inverse_cot(thisdf_copy[var1].values * scaling) * 180 / np.pi
    
    # Calculate angular residuals
    angular_residuals = thisdf_copy['angletrue'] - thisdf_copy['angle']
    
    var1_deg = thisdf_copy['angletrue']
    
    nbins = 15
    xmin = np.min(var1_deg)
    xmax = np.max(var1_deg)
    step = (xmax - xmin) / nbins
    
    # Create binned scatter plot
    sns.regplot(x=var1_deg, y=angular_residuals, x_bins=np.linspace(xmin, xmax, nbins), 
               fit_reg=None, marker='.', ax=ax, scatter_kws={'s': 20})
    ax.set_xlabel('True ' + name)
    ax.set_ylabel('True - predicted ' + name)
    
    # Calculate binned statistics
    bin_centers = []
    means = []
    stds = []
    
    for i in range(nbins):
        bin_mask = (var1_deg >= xmin + i*step) & (var1_deg < xmin + (i+1)*step)
        if np.sum(bin_mask) > 0:
            bin_centers.append(xmin + (i+0.5)*step)
            bin_residuals = angular_residuals[bin_mask]
            means.append(np.mean(bin_residuals))
            stds.append(np.std(bin_residuals))
    
    bin_centers = np.array(bin_centers)
    means = np.array(means)
    stds = np.array(stds)
    
    # Create uncertainty bands
    upbar = means + stds
    downbar = means - stds
    
    ax.fill_between(bin_centers, upbar, downbar, alpha=0.3, color='lightblue')
    
    return ax


In [ ]:
def create_slim_summary_plot_horizontal(df, base_dir):
   """
   Create horizontal summary residual plot for Slim model (1x3 layout)
   """
   fig, axes = plt.subplots(1, 3, figsize=(18, 6))
   fig.tight_layout(pad=4.0)
   
   # X residuals
   residual_plot_slim(axes[0], df, 'xtrue', 'x', r'$x$ [μm]', scaling=SCALE_FACTOR_X)
   axes[0].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[0].axvline(-25, color='gray', linestyle=':', alpha=0.7)
   axes[0].axvline(25, color='gray', linestyle=':', alpha=0.7)
   axes[0].set_ylim([-10, 10])
   
   # Y residuals
   residual_plot_slim(axes[1], df, 'ytrue', 'y', r'$y$ [μm]', scaling=SCALE_FACTOR_Y)
   axes[1].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[1].axvline(-6.25, color='gray', linestyle=':', alpha=0.7)
   axes[1].axvline(6.25, color='gray', linestyle=':', alpha=0.7)
   axes[1].set_ylim([-10, 10])
   
   # CotB residuals (converted to degrees)
   residual_plot_deg_slim(axes[2], df, 'cotBtrue', 'cotB', r'$β$ [deg]', scaling=SCALE_FACTOR_COTB)
   axes[2].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[2].axvline(90, color='gray', linestyle=':', alpha=0.7)
 #  axes[2].set_ylim([-10, 10])
   
   # Save figure
   save_fig_path = os.path.join(base_dir, 'slim_residual_summary_horizontal.png')
   plt.savefig(save_fig_path, dpi=300, bbox_inches='tight')
   plt.show()
   
   return fig

In [ ]:
create_slim_summary_plot_horizontal(df, base_dir)

In [ ]:
def create_slim_summary_plot_horizontal(df, base_dir):
   """
   Create horizontal summary residual plot for Slim model (1x3 layout)
   """
   fig, axes = plt.subplots(1, 3, figsize=(18, 6))
   fig.tight_layout(pad=4.0)
   
   # X residuals
   residual_plot_slim(axes[0], df, 'xtrue', 'x', r'$x$ [μm]', scaling=SCALE_FACTOR_X)
   axes[0].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[0].axvline(-25, color='gray', linestyle=':', alpha=0.7)
   axes[0].axvline(25, color='gray', linestyle=':', alpha=0.7)
   axes[0].set_ylim([-10, 10])
   
   # Y residuals
   residual_plot_slim(axes[1], df, 'ytrue', 'y', r'$y$ [μm]', scaling=SCALE_FACTOR_Y)
   axes[1].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[1].axvline(-6.25, color='gray', linestyle=':', alpha=0.7)
   axes[1].axvline(6.25, color='gray', linestyle=':', alpha=0.7)
   axes[1].set_ylim([-10, 10])
   
   # CotB residuals (converted to degrees)
   residual_plot_deg_slim(axes[2], df, 'cotBtrue', 'cotB', r'$β$ [deg]', scaling=SCALE_FACTOR_COTB)
   axes[2].axhline(0, color='gray', linestyle=':', alpha=0.7)
   axes[2].axvline(90, color='gray', linestyle=':', alpha=0.7)
   axes[2].set_ylim([-10, 10])
   
   # Save figure
   save_fig_path = os.path.join(base_dir, 'slim_residual_summary_horizontal.png')
   plt.savefig(save_fig_path, dpi=300, bbox_inches='tight')
   plt.show()
   
   return fig

In [ ]:
print(os.getcwd())

In [ ]:
#df.to_parquet('/depot/cms/private/users/kuang14/Smart_Pixel/Full_n_slim/Model_performance/Slim_Models/slim_soft_quant_no_drop.parquet')